In [1]:
from pathlib import Path
import zipfile
from zipfile import ZipFile
import warnings 

import pandas as pd 
import numpy as np

from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

import matplotlib.pyplot as plt 
from tqdm.auto import tqdm 
from torchinfo import summary 

import torch
from torch import nn 
import torch.functional as F 
from torch.utils.data import Dataset, DataLoader 

from tfrecord.reader import tfrecord_loader

warnings.filterwarnings("ignore")

In [2]:
SEED = 2026
N_SPLITS = 5
BATCH_SIZE = 32

DEV = torch.device("mps" 
if torch.backends.mps.is_available()
else "cuda"
if torch.cuda.is_available()
else "cpu")

print(f"device: {DEV}")

device: mps


In [3]:
DATA_PATH = Path("data")
FILE_PATH = DATA_PATH / "tpu-getting-started"
FILE_PATH.mkdir(parents=True, exist_ok=True)
FILE_PATH_ZIP = DATA_PATH / "tpu-getting-started.zip"

with zipfile.ZipFile(FILE_PATH_ZIP, "r") as ref_zip:
    print(f"Unzipping flowers data in {FILE_PATH} ...")
    ref_zip.extractall(FILE_PATH)

Unzipping flowers data in data/tpu-getting-started ...


In [4]:
SRC = Path("/data/tpu-getting-started/tfrecords-jpeg-224x224")
DST = Path("data/flowers-224")

def convert_split(split: str, labeled: bool = True):

    out_dir = DST / split
    out_dir.mkdir(parents=True, exist_ok=True)

    description = {"image": "byte", "id": "byte"}

    if labeled:
        description["class"] = "int"

    rows = []
    for tfrec in sorted((SRC / split).glob("*.tfrec")):
        for rec in tqdm(tfrecord_loader(str(tfrec), None, description), desc=tfrec.name):

            img_id = rec["id"].tobytes().decode()
            (out_dir / f"{img_id}.jpg").write_bytes(rec["image"].tobytes())

            row = {"id": img_id}
            if labeled:
                row["label"] = int(rec["class"][0])

            rows.append(row)

    pd.DataFrame(rows).to_csv(DST / f"{split}.csv", index=False)
    print(len(rows))